# Run pipeline
Finds every game with data in bronze and runs bronze to silver to gold for each one.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import yaml
from pyspark.sql import SparkSession

sys.path.append("..")
from utility import discover_games, layer_path, process_game

spark = SparkSession.builder.master("local[*]").appName("run_pipeline").getOrCreate()

# Parameter

In [ ]:
cfg = yaml.safe_load(open("../config/keyplayer_battle.yaml", encoding="utf-8"))
BASE = Path("..") / cfg["data"]["root"]

ONLY_NEW = False      # True = skip games that already have gold

# Find Games

In [ ]:
games = discover_games(cfg, BASE)
if ONLY_NEW:
    games = [(y, g) for y, g in games
             if not layer_path(cfg, BASE, "gold", "player_metrics", y, g).exists()]
print(f"{len(games)} game(s) to process:", games)

# Run

In [ ]:
results = []
for year, game in games:
    try:
        results.append({**process_game(spark, cfg, BASE, year, game), "status": "ok"})
    except Exception as error:                     # one broken game shouldn't stop the others
        results.append({"year": year, "game": game, "status": f"failed: {error}"})
    print(results[-1])

# Summary

In [ ]:
summary = pd.DataFrame(results)
summary